In [1]:
#!/usr/bin/env python3

from typing import Iterable, List, Tuple
import re
import onnxruntime as ort
import torch
import numpy as np
import torch.nn as nn
import torch
from melo import utils
from melo.models import SynthesizerTrn
from melo.split_utils import split_sentence
from melo.download_utils import load_or_download_config, load_or_download_model


c:\Users\MYSTIC Ganesh\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [2]:
def text_split(text, desired_length=100, max_length=200):
    """Split text it into chunks of a desired length trying to keep sentences intact."""
    text = re.sub(r'\n\n+', '\n', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[""]', '"', text)
    text = re.sub(r'([,.?!])', r'\1 ', text)
    text = re.sub(r'\s+', ' ', text)

    rv = []
    in_quote = False
    current = ""
    split_pos = []
    pos = -1
    end_pos = len(text) - 1

    def seek(delta):
        nonlocal pos, in_quote, current
        is_neg = delta < 0
        for _ in range(abs(delta)):
            if is_neg:
                pos -= 1
                current = current[:-1]
            else:
                pos += 1
                current += text[pos]
            if text[pos] == '"':
                in_quote = not in_quote
        return text[pos]

    def peek(delta):
        p = pos + delta
        return text[p] if p < end_pos and p >= 0 else ""

    def commit():
        nonlocal rv, current, split_pos
        rv.append(current)
        current = ""
        split_pos = []
    while pos < end_pos:
        c = seek(1)
        if len(current) >= max_length:
            if len(split_pos) > 0 and len(current) > (desired_length / 2):
                d = pos - split_pos[-1]
                seek(-d)
            else:
                while c not in '!?.\n ' and pos > 0 and len(current) > desired_length:
                    c = seek(-1)
            commit()
        elif not in_quote and (c in '!?\n' or (c in '.,' and peek(1) in '\n ')):
            while pos < len(text) - 1 and len(current) < max_length and peek(1) in '!?.':
                c = seek(1)
            split_pos.append(pos)
            if len(current) >= desired_length:
                commit()
        elif in_quote and peek(1) == '"' and peek(2) in '\n ':
            seek(2)
            split_pos.append(pos)
    rv.append(current)
    rv = [s.strip() for s in rv]
    rv = [s for s in rv if len(s) > 0 and not re.match(r'^[\s\.,;:!?]*$', s)]
    return rv


def split_sentence(text, min_len=10):
    text = re.sub('[。！？；]', '.', text)
    text = re.sub('[，]', ',', text)
    text = re.sub('[“”]', '"', text)
    text = re.sub('[‘’]', "'", text)
    text = re.sub(r"[\<\>\(\)\[\]\"\«\»]+", "", text)
    return [item.strip() for item in text_split(text, 256, 512) if item.strip()]


def split_sentences_into_pieces(text,  quiet=False):
    texts = split_sentence(text, )
    if not quiet:
        print(" > Text split to sentences.")
        print("\n".join(texts))
        print(" > ===========================")
    return texts


In [ ]:

class OnnxModel:
    def __init__(self, filename):
        session_opts = ort.SessionOptions()
        session_opts.inter_op_num_threads = 1
        session_opts.intra_op_num_threads = 4

        self.session_opts = session_opts
        self.model = ort.InferenceSession(
            filename,
            # sess_options=self.session_opts,
            providers=["CPUExecutionProvider"],
        )
        meta = self.model.get_modelmeta().custom_metadata_map
        self.bert_dim = int(meta["bert_dim"])
        self.ja_bert_dim = int(meta["ja_bert_dim"])
        self.add_blank = int(meta["add_blank"])
        self.sample_rate = int(meta["sample_rate"])
        self.speaker_id = int(meta["speaker_id"])
        self.lang_id = int(meta["lang_id"])
        self.sample_rate = int(meta["sample_rate"])

    def __call__(self, x, tones):
        """
        Args:
          x: 1-D int64 torch tensor
          tones: 1-D int64 torch tensor
        """
        x = x.unsqueeze(0)
        tones = tones.unsqueeze(0)
        sid = torch.tensor([self.speaker_id], dtype=torch.int64)
        noise_scale = torch.tensor([0.6], dtype=torch.float32)
        length_scale = torch.tensor([1.0], dtype=torch.float32)
        noise_scale_w = torch.tensor([0.8], dtype=torch.float32)

        x_lengths = torch.tensor([x.shape[-1]], dtype=torch.int64)

        y = self.model.run(
            ["y"],
            {
                "x": x.numpy(),
                "tones": tones.numpy(),

                "x_lengths": x_lengths.numpy(),
                "sid": sid.numpy(),
                "noise_scale": noise_scale.numpy(),
                "noise_scale_w": noise_scale_w.numpy(),
                "length_scale": length_scale.numpy(),
            },
        )[0][0][0]
        return y


model = OnnxModel("./onnx/melo.onnx")


In [16]:
# American accent
text = """
Ha!, I knew what you mean?
"""
device = "cpu"
language = 'EN'
hps = load_or_download_config(language, use_hf=True, config_path=None)
output_path = 'en-us.wav'
speaker_ids = hps.data.spk2id
speaker_id = speaker_ids['EN-US']
output_path = "en-us.wav"
sdp_ratio: float = 0.2
noise_scale: float = 0.6
noise_scale_w: float = 0.8
speed: float = 1.0
quiet: bool = False
texts = split_sentences_into_pieces(text,  quiet)
audio_list = []
t = texts[0]
t = re.sub(r"([a-z])([A-Z])", r"\1 \2", t)
symbols = hps.symbols
symbol_to_id = {s: i for i, s in enumerate(symbols)}
bert, ja_bert, phones, tones, lang_ids = utils.get_text_for_tts_infer(
    t, language, hps, device, symbol_to_id
)
# audio_list = []
# with torch.no_grad():
#     x_tst = phones.to(device).unsqueeze(0)
#     tones = tones.to(device).unsqueeze(0)
#     lang_ids = lang_ids.to(device).unsqueeze(0)
#     bert = bert.to(device).unsqueeze(0)
#     ja_bert = ja_bert.to(device).unsqueeze(0)
#     x_tst_lengths = torch.LongTensor([phones.size(0)]).to(device)
#     del phones
#     speakers = torch.LongTensor([speaker_id]).to(device)


None
 > Text split to sentences.
Ha! , I knew what you mean?
 > ===========================


In [17]:
import soundfile as sf
y = model(x=phones, tones=tones)

sf.write("./test.wav", y, model.sample_rate)


In [18]:

from IPython.display import Audio
# Display the audio player
Audio("./test.wav", autoplay=True)
